# Real-time labor-market deterioration

This notebook asks how cross-asset outcomes differ after a real-time unemployment
deterioration alert. The alert is Sahm-rule-like, but it is calculated directly from
the vintage unemployment rate and is not the official recession indicator.

The design was fixed before results were retrieved. Live outputs are temporary. The
committed notebook explains the method without storing provider observations,
figures, tables, or empirical conclusions.

**Primary protocol.** Each unit is the last common ETF trading session of a complete
month, and the information cutoff is one calendar day earlier. The treatment is a
deterioration alert; the reference is no alert; the primary horizon is one calendar
month. The four primary outcomes are `SPY`, `DBC`, `TLT`, and the paired return spread
`TLT-SHY`. Expected directions are negative for the two risk-asset contrasts and
positive for the Treasury contrasts. Bonferroni intervals cover this one-month family.
Longer horizons, alternative alert boundaries, revised history, and stability splits
are exploratory.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from studies._support import (
    CORE_SYMBOLS,
    STUDY_START,
    acquire_latest_series,
    acquire_monthly_prices,
    acquire_vintage_histories,
    assert_component_periods_match,
    build_point_in_time_levels,
    classification_transition_table,
    combined_outcome_labels,
    compare_statistics,
    configure_plots,
    familywise_primary_intervals,
    feature_provenance_summary,
    forward_labels,
    latest_revised_counterpart,
    latest_revised_labor_deterioration,
    momentum_baseline,
    open_live_session,
    plot_coverage,
    plot_feature_comparison,
    plot_normalized_prices,
    plot_regime_contrasts,
    plot_regime_distributions,
    plot_regime_means,
    plot_regime_timeline,
    plot_revision_gap,
    plot_sample_sizes,
    plot_sensitivity_heatmap,
    point_in_time_labor_deterioration,
    regime_contrast_statistics,
    regime_statistics,
    return_spread_labels,
    simultaneous_interval_family,
    study_run_manifest,
    temporal_contrast_stability,
    unconditional_statistics,
    validate_study_outputs,
)

configure_plots()
pd.set_option("display.max_columns", 20)
session = open_live_session()

## Hypothesis, assets, and source series

The core cross-asset proxies are extended with `TLT` and `SHY`. The additions separate
long- and short-duration Treasury behavior when labor conditions weaken. The fixed ETF
set is liquid and interpretable, but it is not survivorship-free.

`UNRATE` is a monthly, seasonally adjusted unemployment rate that can be revised. The
hypothesis is that equity and commodity outcomes are weaker, and high-quality bond
outcomes stronger, after an alert. This is an association hypothesis. Labor releases
do not cause every subsequent market move, and an alert is not a deterministic market
timing signal.

`TLT-SHY` is a within-date difference of two forward returns, not the difference of
separately estimated tables. It retains the covariance between long- and short-duration
Treasuries and directly asks whether maturity exposure behaves differently during an
alert. `IEF` remains visible as an intermediate-duration reference. Adjusted closes
capture provider adjustments but omit spreads, taxes, and any execution rule.

In [ ]:
symbols = (*CORE_SYMBOLS, "TLT", "SHY")
price_history, market_provenance = acquire_monthly_prices(session, symbols)
prices = price_history.loc[STUDY_START:]
histories = acquire_vintage_histories(session, ("UNRATE",), prices.index)
latest = acquire_latest_series(session, ("UNRATE",))
staleness = {"UNRATE": pd.Timedelta(days=62)}
point_in_time = build_point_in_time_levels(
    histories,
    prices.index,
    staleness,
)
latest_levels = latest_revised_counterpart(point_in_time, latest)
feature_provenance = feature_provenance_summary(point_in_time)
manifest = study_run_manifest(
    prices,
    series_ids=("UNRATE",),
    thresholds="labor alert=0.5 percentage points",
    staleness=staleness,
)
display(manifest, market_provenance, feature_provenance)

## Coverage and release staleness

Month-end decisions often use an unemployment observation for an earlier reference
month because the current month's survey has not been released. That is correct
point-in-time behavior. The maximum-staleness rule permits ordinary publication delay
but rejects a feature if releases stop arriving. Missing or deleted source values stay
missing rather than falling back to a more favorable older observation.

Observation month, source availability, and retrieval time are distinct. The code
selects only a version whose daily availability interval contains the lagged cutoff.
It rejects matches older than 62 days. Market and macro results are saved to and
reloaded from a temporary DuckDB database before analysis, verifying the storage
boundary without publishing a snapshot. The coverage table should be read before the
signal because a valid current level does not guarantee all 15 required source months.

In [ ]:
figure, _ = plot_coverage(market_provenance, feature_provenance)
plt.show()
plt.close(figure)

## Point-in-time unemployment versus latest-revised history

Persistra selects the source version whose availability interval contains the lagged
decision date. The counterfactual latest-revised view retains the same selected
observation month and substitutes today's value. That comparison isolates revisions;
it does not pretend the observation itself was published earlier.

The level plot is an audit of information timing. Small visual differences can still
matter near a fixed alert threshold, so classification changes are examined later.

In [ ]:
point_levels = point_in_time.frame.rename(columns={"UNRATE": "Unemployment rate"})
revised_levels = latest_levels.set_axis(point_levels.columns, axis="columns")
figure, _ = plot_feature_comparison(
    point_levels,
    revised_levels,
    ("Unemployment rate",),
)
plt.show()
plt.close(figure)

## Alert construction

At each decision, the code opens one as-of unemployment vintage and requests 15 exact,
consecutive observation months. Let \(u_{d,t}\) be the unemployment rate for source
month \(t\) in the vintage known at cutoff \(d\), and let
\(m_{d,t}=(u_{d,t}+u_{d,t-1}+u_{d,t-2})/3\). The signal is
\(m_{d,t}-\min(m_{d,t-1},\ldots,m_{d,t-12})\). The current mean is excluded from its
own baseline by construction. An alert begins at 0.5 percentage points.

This resembles the intuition of the Sahm rule but is deliberately calculated from the
selected `UNRATE` vintage and is not the official indicator. The component plot shows
the latest level, current three-month mean, and prior twelve-month low so the reader
can audit what moves the gap. Exact source-month provenance prevents repeated decision
rows from standing in for distinct labor observations.

In [ ]:
def labor_regime(signal: pd.Series, *, boundary: float = 0.5) -> pd.Series:
    regime = pd.Series(pd.NA, index=signal.index, dtype="string")
    regime.loc[signal.notna() & signal.lt(boundary)] = "no alert"
    regime.loc[signal.notna() & signal.ge(boundary)] = "deterioration alert"
    return regime

point_signal_result = point_in_time_labor_deterioration(
    histories["UNRATE"], point_in_time
)
latest_signal_result = latest_revised_labor_deterioration(
    point_in_time, latest["UNRATE"]
)
point_signal = point_signal_result.frame["UNRATE"]
latest_signal = latest_signal_result.frame["UNRATE"]
point_regimes = labor_regime(point_signal)
latest_regimes = labor_regime(latest_signal)
display(point_regimes.value_counts(dropna=False).rename("decision count"))
figure, _ = plot_regime_timeline(
    point_signal,
    point_regimes,
    title="Real-time labor deterioration signal",
    ylabel="Percentage-point increase",
    boundaries=(0.5,),
)
plt.show()
plt.close(figure)

component_values = (
    point_signal_result.provenance.loc[
        point_signal_result.provenance["view"].eq("point-in-time")
    ]
    .pivot(index="decision_date", columns="component", values="value")
    .astype("Float64")
)
ordered_components = [f"month_{offset}" for offset in range(-14, 1)]
smoothed = (
    component_values[ordered_components]
    .T.rolling(3, min_periods=3)
    .mean()
    .T
)
labor_components = pd.DataFrame(
    {
        "latest unemployment rate": component_values["month_0"],
        "current three-month mean": smoothed["month_0"],
        "prior twelve-month low": smoothed[
            [f"month_{offset}" for offset in range(-12, 0)]
        ].min(axis=1),
    }
)
figure, axis = plt.subplots(figsize=(12, 5.5))
for column, style in zip(
    labor_components.columns,
    ("-", "--", ":"),
    strict=True,
):
    axis.plot(
        labor_components.index,
        labor_components[column],
        label=column,
        linestyle=style,
    )
axis.set(
    title="Components of the real-time labor alert",
    ylabel="Unemployment rate (percentage points)",
)
axis.legend()
figure.tight_layout()
plt.show()
plt.close(figure)

## Labels and baselines

One-, three-, and twelve-month forward returns are separate label objects with explicit
end dates. The notebook never joins a future label into the unemployment feature
panel. The unconditional baseline shows the macro-eligible distribution. A trailing
twelve-month price-momentum split supplies a familiar asset-only time-series baseline
that does not depend on macro revisions.

Normalized prices are context, not a backtest. There are no portfolio weights,
rebalancing rules, costs, or claims about investability.

Forward return \(R_{d,h}=P_{d+h}/P_d-1\) lives in a typed label object with an actual
ending close. The calendar period of that close must be exactly \(h\) months after the
decision. The unconditional baseline is restricted to dates with a valid labor state.
A one-year price warm-up makes the momentum split available from the beginning of the
analysis window, so differences are not driven by a hidden baseline burn-in.

In [ ]:
labels = forward_labels(prices)
eligible = point_regimes.notna()
unconditional = unconditional_statistics(labels, eligible=eligible)
momentum = momentum_baseline(price_history, labels, eligible=eligible)
display(unconditional, momentum)
figure, _ = plot_normalized_prices(prices)
plt.show()
plt.close(figure)

## Uncertainty and conditional summaries

The summaries retain counts, coverage, means, volatility, positive shares, and
heteroskedasticity-and-autocorrelation-consistent (HAC) intervals. The horizon-aware
lag addresses overlap in multi-month labels but cannot
create independent alert episodes. The one-month maximum drawdown resets when an
alert changes or an observation is missing, preventing disconnected alert periods
from being compounded as one path.

Interpret wide or overlapping intervals as uncertainty, not as evidence that two
economic states are equivalent.

The explicit alert-minus-no-alert estimator is a regression contrast whose HAC score
remains aligned to the full monthly calendar. Its bandwidth is at least \(h-1\) for
overlapping labels and may be longer under the automatic rule. Descriptive regime-mean
intervals are pointwise. Only the four predeclared one-month contrasts use simultaneous
Bonferroni intervals. The minimum-data flag requires twelve outcomes and two
outcome-eligible episodes on each side; it is a display threshold, not proof that a
two-episode normal approximation is trustworthy. Leave-one-episode-out estimates are
essential context when alerts cluster.

In [ ]:
point_statistics = regime_statistics(labels, point_regimes)
latest_statistics = regime_statistics(labels, latest_regimes)
directional_contrasts = regime_contrast_statistics(
    labels,
    point_regimes,
    treated="deterioration alert",
    reference="no alert",
    assets=("SPY", "DBC", "TLT"),
)
duration_labels = return_spread_labels(
    labels, {"TLT minus SHY": ("TLT", "SHY")}
)
duration_contrast = regime_contrast_statistics(
    duration_labels,
    point_regimes,
    treated="deterioration alert",
    reference="no alert",
)
primary_contrasts = familywise_primary_intervals(
    pd.concat([directional_contrasts, duration_contrast], ignore_index=True)
)
display(point_statistics, primary_contrasts)
figure, _ = plot_regime_means(
    point_statistics,
    title="Labor deterioration and one-month outcomes",
)
plt.show()
plt.close(figure)

figure, _ = plot_regime_contrasts(
    primary_contrasts,
    title="Predeclared labor-alert contrasts",
)
plt.show()
plt.close(figure)

## Distribution shape and rare episodes

Labor alerts are expected to cluster around a limited number of stress episodes.
Means can therefore conceal skew, outliers, and unequal sample sizes. The box plots
compare the central distributions for risk assets and Treasury durations. The count
chart shows whether an apparent difference rests on a small alert sample.

Removing outliers after seeing them would change the analysis plan, so the numerical
summaries retain every finite provider observation.

Box plots show quartiles and whiskers while retaining every tail observation as a
plotted flier. They are not density estimates. The companion bars distinguish outcome
months from contiguous labor episodes after unavailable labels are masked. This avoids
counting a terminal alert episode that contributes no return as inferential support.

In [ ]:
figure, _ = plot_regime_distributions(
    labels[1],
    point_regimes,
    assets=("SPY", "DBC", "TLT", "SHY"),
    title="One-month outcomes with and without a labor alert",
)
plt.show()
plt.close(figure)

figure, _ = plot_sample_sizes(point_statistics)
plt.show()
plt.close(figure)

## Sensitivity to the alert boundary

The predeclared boundaries are 0.3, 0.5, and 0.7 percentage points. For every asset,
the heatmap reports the one-month mean during alerts minus the mean without an alert.
It displays the full asset-by-threshold family instead of selecting the most favorable
asset or boundary. A stable pattern should survive nearby definitions; a fragile one
is a limitation to report.

Every threshold produces a long contrast table with counts, episode counts, HAC errors,
and nominal intervals. One exploratory Bonferroni family is then recomputed across the
complete threshold-by-asset display. Gray or masked cells fail the predeclared
minimum-data rule; they are neither zero nor evidence of no association.

In [ ]:
sensitivity_rows = []
for boundary in (0.3, 0.5, 0.7):
    table = regime_contrast_statistics(
        {1: labels[1]},
        labor_regime(point_signal, boundary=boundary),
        treated="deterioration alert",
        reference="no alert",
        assets=symbols,
    )
    table.insert(0, "boundary", boundary)
    sensitivity_rows.append(table)
sensitivity_table = simultaneous_interval_family(
    pd.concat(sensitivity_rows, ignore_index=True),
    family_id="labor threshold family",
)
sensitivity = sensitivity_table.pivot(
    index="boundary", columns="asset", values="mean_difference"
).where(
    sensitivity_table.pivot(
        index="boundary", columns="asset", values="meets_display_threshold"
    )
)
display(sensitivity_table, sensitivity)
figure, _ = plot_sensitivity_heatmap(
    sensitivity,
    title="Alert-minus-no-alert mean return across fixed boundaries",
    color_label="Difference in one-month mean return",
)
plt.show()
plt.close(figure)

## Latest-revised classification bias

Revision effects enter through two channels: the level can change, and a small change
can move a decision across the alert boundary. The comparison table preserves both
membership and mean differences. The plotted signal gap is retrospective and remains
outside the primary feature set.

Agreement between real-time and latest-revised labels in this sample would be a
legitimate negative result, not a reason to omit the comparison.

Latest-revised signal components use the exact same 15 source months as the real-time
signal. The classification cross-tab includes unclassified dates, so availability
changes are not hidden inside a mean difference. The first/second-half and
leave-one-episode-out table uses the real-time classification only and should be read
as temporal robustness, not as evidence about revision bias.

In [ ]:
revision_comparison = compare_statistics(point_statistics, latest_statistics)
classification_changes = classification_transition_table(
    point_regimes,
    latest_regimes,
)
stability_labels = combined_outcome_labels(
    labels,
    assets=("SPY", "DBC", "TLT"),
    spreads={"TLT minus SHY": ("TLT", "SHY")},
)
stability = temporal_contrast_stability(
    stability_labels,
    point_regimes,
    treated="deterioration alert",
    reference="no alert",
)
display(revision_comparison, classification_changes, stability)
figure, _ = plot_revision_gap(
    point_signal,
    latest_signal,
    title="Labor-signal revision substitution gap",
)
plt.show()
plt.close(figure)

## Limitations and executable review

The alert is monthly, release-lagged, and based on a single national series. It ignores
intramonth information, labor-force composition, and other recession evidence. The
contemporary ETF universe has inception and survivorship bias. Returns exclude costs,
spreads, taxes, and implementation delay. Alert months cluster, secondary horizons
overlap, normal intervals are approximate, and repeated assets and thresholds raise
multiple-testing risk.

Mechanical assertions verify causal availability, label separation, alignment,
finite outputs, and provenance coverage. They do not certify the hypothesis.

The 0.5-point rule is a research definition, not an official recession declaration.
The [real-time Sahm indicator page](https://fred.stlouisfed.org/release?rid=456) gives
context for the official concept, while the
[ALFRED guide](https://fred.stlouisfed.org/docs/api/fred/realtime_period.html) explains
historical information sets. The HAC intervals follow the covariance construction in
[Newey and West (1987)](https://doi.org/10.2307/1913610). These references motivate the
concepts; they do not validate this ETF association design.

In [ ]:
audit = validate_study_outputs(
    prices,
    point_in_time,
    labels,
    point_regimes,
    point_statistics,
    expected_regimes=frozenset({"no alert", "deterioration alert"}),
    transformed=point_signal_result,
)
assert_component_periods_match(point_signal_result, latest_signal_result)
assert set(point_in_time.frame.columns).isdisjoint(labels[1].frame.columns)
matched = point_in_time.provenance["available_from"].notna()
assert point_in_time.provenance.loc[matched, "available_from"].le(
    point_in_time.provenance.loc[matched, "decision_date"] - pd.Timedelta(days=1)
).all()
display(audit)
session.close()

## Interpretation after execution

Begin with coverage, alert counts, and episode concentration. Compare full
distributions and uncertainty with both baselines, then inspect threshold stability
and latest-revised classification changes. Keep results that are null, unstable, or
opposite the hypothesis. The notebook measures association only; it does not establish
recession timing, causality, or a profitable defensive rotation.